# 🧠 Clase 7 — Tu primera red neuronal con PyTorch

Ya sabés cargar datos, limpiarlos, particionarlos y evaluarlos.
Ahora vas a construir un modelo que **aprende solo** a partir de los datos.

### ¿Qué vamos a cubrir?

| Sección | Concepto |
|---|---|
| 1 | Setup: instalar e importar PyTorch |
| 2 | La neurona artificial |
| 3 | Funciones de activación |
| 4 | Tu primera red: el MLP |
| 5 | La función de pérdida |
| 6 | Backpropagation: ¿cómo aprende? |
| 7 | Loop de entrenamiento completo |
| 8 | Curvas de pérdida y diagnóstico |
| 9 | Experimentos con hiperparámetros |
| 10 | Ejercicio evaluable |

> **Hoy pasás de espectador a entrenador de redes neuronales.**

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import random
import math

# Semilla para reproducibilidad
torch.manual_seed(42)
random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Dispositivo: CPU (sin GPU necesaria)")
print("✅ Entorno listo.")

---
## 2. La neurona artificial

Una neurona artificial es la **unidad básica** de una red neuronal.
Funciona en 3 pasos:

```
Entradas (x₁, x₂, ...) → Suma ponderada (z) → Activación (a) → Salida
```

### Fórmula

$z = w_1 \cdot x_1 + w_2 \cdot x_2 + ... + w_n \cdot x_n + b$

$a = f(z)$

| Componente | ¿Qué es? | Analogía |
|---|---|---|
| **x** | Entradas (features) | Las notas de un estudiante |
| **w** | Pesos (parámetros) | La importancia de cada nota |
| **b** | Sesgo (bias) | Un ajuste base |
| **f** | Función de activación | La decisión final |

In [ ]:
# Una neurona desde cero (sin PyTorch, para entender)

def neurona(entradas, pesos, sesgo):
    """Una neurona artificial: suma ponderada + activación."""
    # Paso 1: suma ponderada
    z = sum(x * w for x, w in zip(entradas, pesos)) + sesgo
    
    # Paso 2: activación (ReLU)
    a = max(0, z)  # ReLU: si z < 0 → 0, si z >= 0 → z
    
    return z, a

# Ejemplo: ¿aprueba el estudiante?
entradas = [7.0, 0.8]       # nota_parcial=7.0, asistencia=0.8 (80%)
pesos    = [0.6, 3.0]       # la asistencia importa más (x3)
sesgo    = -5.0             # umbral base

z, a = neurona(entradas, pesos, sesgo)
print(f"Entradas:  {entradas}")
print(f"Pesos:     {pesos}")
print(f"Sesgo:     {sesgo}")
print(f"")
print(f"Suma ponderada (z): {entradas[0]}*{pesos[0]} + {entradas[1]}*{pesos[1]} + ({sesgo}) = {z:.1f}")
print(f"Activación ReLU(z): max(0, {z:.1f}) = {a:.1f}")
print(f"")
print(f"💡 La salida es {a:.1f}. Si fuera 0, indicaría 'no aprueba'.")

In [ ]:
# La misma neurona con PyTorch

# nn.Linear(2, 1) = una capa con 2 entradas → 1 salida
# Es exactamente una neurona con 2 pesos + 1 sesgo

capa = nn.Linear(2, 1)

# Asignar los mismos pesos del ejemplo anterior
with torch.no_grad():
    capa.weight[0][0] = 0.6
    capa.weight[0][1] = 3.0
    capa.bias[0]      = -5.0

# Forward pass
x = torch.tensor([7.0, 0.8])
z = capa(x)                          # suma ponderada
a = torch.relu(z)                    # activación

print(f"PyTorch z: {z.item():.1f}")
print(f"PyTorch a: {a.item():.1f}")
print(f"")
print(f"📌 ¡Mismo resultado! PyTorch hace lo mismo pero más rápido y con autograd.")

---
## 3. Funciones de activación

Sin activación, una red neuronal sería solo una multiplicación de matrices (=regresión lineal).
Las activaciones le dan **no-linealidad**: la capacidad de aprender patrones complejos.

### Las 3 activaciones más usadas

| Activación | Fórmula | Rango | ¿Cuándo usarla? |
|---|---|---|---|
| **ReLU** | $\max(0, z)$ | $[0, +\infty)$ | Capas ocultas (default) |
| **Sigmoid** | $\frac{1}{1+e^{-z}}$ | $(0, 1)$ | Salida binaria (probabilidad) |
| **Tanh** | $\frac{e^z - e^{-z}}{e^z + e^{-z}}$ | $(-1, 1)$ | Capas ocultas (alternativa) |

In [ ]:
# Visualización de las 3 activaciones

x = torch.linspace(-5, 5, 200)

activaciones = {
    "ReLU":    torch.relu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh":    torch.tanh(x),
}

colores = {"ReLU": "#e74c3c", "Sigmoid": "#3498db", "Tanh": "#2ecc71"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, y) in zip(axes, activaciones.items()):
    ax.plot(x.numpy(), y.numpy(), color=colores[nombre], linewidth=2.5)
    ax.axhline(y=0, color="gray", linewidth=0.5)
    ax.axvline(x=0, color="gray", linewidth=0.5)
    ax.set_title(nombre, fontsize=13, fontweight="bold")
    ax.grid(True, alpha=0.2)
    ax.set_xlabel("z", fontsize=11)
    ax.set_ylabel(f"{nombre}(z)", fontsize=11)

fig.suptitle("Funciones de activación: la clave de las redes neuronales",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("💡 ReLU es la más popular para capas ocultas: simple, rápida, efectiva.")
print("   Sigmoid se usa en la última capa para clasificación binaria (P(spam)=0.87).")

---
## 4. Tu primera red: el MLP (Perceptrón Multicapa)

Un MLP apila capas de neuronas para aprender patrones complejos.

```
Entrada (5 features)
    │
    ▼
┌──────────────┐
│  Capa oculta  │  16 neuronas + ReLU
└──────────────┘
    │
    ▼
┌──────────────┐
│  Capa oculta  │  8 neuronas + ReLU
└──────────────┘
    │
    ▼
┌──────────────┐
│  Capa salida  │  1 neurona (predicción)
└──────────────┘
```

> 💡 "Profundo" (Deep Learning) = muchas capas apiladas.

In [ ]:
# Generar dataset sintético: predecir nota_final a partir de features del estudiante

random.seed(42)
N = 200

# Features: horas_estudio, asistencia_pct, nota_parcial, horas_sueño, motivacion
datos = []
for _ in range(N):
    horas_estudio  = random.uniform(1, 20)
    asistencia     = random.uniform(0.4, 1.0)
    nota_parcial   = random.uniform(2, 10)
    horas_sueno    = random.uniform(4, 9)
    motivacion     = random.uniform(1, 10)
    
    # Nota final (relación real con algo de ruido)
    nota_final = (
        0.15 * horas_estudio +
        2.0 * asistencia +
        0.4 * nota_parcial +
        0.1 * horas_sueno +
        0.05 * motivacion +
        random.gauss(0, 0.5)
    )
    nota_final = max(1, min(10, nota_final))  # recortar entre 1 y 10
    
    datos.append({
        "features": [horas_estudio, asistencia, nota_parcial, horas_sueno, motivacion],
        "target": nota_final,
    })

# Convertir a tensores de PyTorch
X = torch.tensor([d["features"] for d in datos], dtype=torch.float32)
y = torch.tensor([d["target"]   for d in datos], dtype=torch.float32).unsqueeze(1)

print(f"📊 Dataset: {X.shape[0]} estudiantes, {X.shape[1]} features")
print(f"   X shape: {X.shape}  (filas × features)")
print(f"   y shape: {y.shape}  (filas × 1)")
print(f"   y rango: [{y.min():.1f}, {y.max():.1f}]")

In [ ]:
# Partición train/val/test (70/15/15)

indices = list(range(N))
random.shuffle(indices)

n_train = int(0.70 * N)  # 140
n_val   = int(0.15 * N)  #  30

idx_train = indices[:n_train]
idx_val   = indices[n_train:n_train + n_val]
idx_test  = indices[n_train + n_val:]

X_train, y_train = X[idx_train], y[idx_train]
X_val,   y_val   = X[idx_val],   y[idx_val]
X_test,  y_test  = X[idx_test],  y[idx_test]

print(f"Train:      {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test:       {len(X_test)} muestras")

In [ ]:
# Normalización Min-Max
# IMPORTANTE: calcular min/max SOLO con datos de train

X_min = X_train.min(dim=0).values
X_max = X_train.max(dim=0).values

def normalizar(X, X_min, X_max):
    return (X - X_min) / (X_max - X_min + 1e-8)  # +1e-8 para evitar /0

X_train_n = normalizar(X_train, X_min, X_max)
X_val_n   = normalizar(X_val,   X_min, X_max)
X_test_n  = normalizar(X_test,  X_min, X_max)

print("✅ Features normalizadas [0, 1] usando min/max del train.")
print(f"   Antes: X_train[0] = [{', '.join(f'{v:.1f}' for v in X_train[0].tolist())}]")
print(f"   Después: X_train_n[0] = [{', '.join(f'{v:.3f}' for v in X_train_n[0].tolist())}]")

In [ ]:
# Definir el MLP con PyTorch

class MLP(nn.Module):
    def __init__(self, n_features, hidden1, hidden2):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_features, hidden1),  # entrada → capa oculta 1
            nn.ReLU(),                        # activación
            nn.Linear(hidden1, hidden2),      # capa oculta 1 → capa oculta 2
            nn.ReLU(),                        # activación
            nn.Linear(hidden2, 1),            # capa oculta 2 → salida
        )
    
    def forward(self, x):
        return self.red(x)

modelo = MLP(n_features=5, hidden1=16, hidden2=8)

# Contar parámetros
n_params = sum(p.numel() for p in modelo.parameters())
print(f"📊 Arquitectura del MLP:")
print(f"   Entrada:   5 features")
print(f"   Oculta 1:  16 neuronas + ReLU")
print(f"   Oculta 2:   8 neuronas + ReLU")
print(f"   Salida:     1 neurona")
print(f"   Total parámetros: {n_params}")
print()
print(modelo)

---
## 5. La función de pérdida (Loss)

La **loss** mide qué tan mal predice el modelo. El objetivo del entrenamiento es **minimizarla**.

| Problema | Loss típica | Fórmula |
|---|---|---|
| Regresión | MSE (Mean Squared Error) | $\frac{1}{n}\sum(y_i - \hat{y}_i)^2$ |
| Clasificación | CrossEntropy | $-\sum y_i \log(\hat{y}_i)$ |

Nosotros usamos **MSE** porque estamos prediciendo una nota (valor continuo).

In [ ]:
# Demostración de la loss

criterio = nn.MSELoss()

# Ejemplo con predicciones buenas vs malas
y_ejemplo = torch.tensor([7.0, 5.0, 8.0])

pred_buena = torch.tensor([6.8, 5.2, 7.9])  # cerca
pred_mala  = torch.tensor([3.0, 9.0, 2.0])   # lejos

loss_buena = criterio(pred_buena, y_ejemplo)
loss_mala  = criterio(pred_mala, y_ejemplo)

print(f"Loss predicción buena: {loss_buena.item():.4f}")
print(f"Loss predicción mala:  {loss_mala.item():.4f}")
print(f"")
print(f"📌 La loss mala es {loss_mala.item()/loss_buena.item():.0f}x más alta.")
print(f"   El entrenamiento busca minimizar este número.")

---
## 6. Backpropagation: ¿cómo aprende la red?

El entrenamiento tiene 3 pasos que se repiten muchas veces:

```
  ┌──────────────────────────────────────────────┐
  │ 1. FORWARD: hacer predicción con pesos actuales │
  │ 2. LOSS: calcular qué tan lejos están           │
  │ 3. BACKWARD: ajustar pesos para reducir la loss │
  └──────────────────────────────────────────────┘
       ↑                                          │
       └──────────── repetir N épocas ────────────┘
```

### ¿Qué es el learning rate?

| Learning rate | Resultado |
|---|---|
| Muy alto (0.1) | El modelo "salta" y no converge |
| Muy bajo (0.00001) | El modelo aprende pero tardísimo |
| Justo (0.001-0.01) | Converge bien ✅ |

> 💡 El learning rate es el hiperparámetro más importante. Si tu modelo no entrena, probá cambiar el LR.

---
## 7. Loop de entrenamiento completo

Ahora juntamos todo: modelo + loss + optimizador + datos → entrenamiento.

In [ ]:
# Loop de entrenamiento paso a paso

# Recrear el modelo con semilla fija
torch.manual_seed(42)
modelo = MLP(n_features=5, hidden1=16, hidden2=8)

# Configuración
criterio    = nn.MSELoss()                               # función de pérdida
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)  # optimizador
n_epocas    = 200                                        # cuántas vueltas

# Guardar historial para graficación
historial = {"train_loss": [], "val_loss": []}

print("🚀 Entrenando...\n")
print(f"{'Época':<8} {'Train Loss':<14} {'Val Loss':<14} {'Status'}")
print("─" * 50)

for epoca in range(1, n_epocas + 1):
    # ── 1. FORWARD: predicción ──
    modelo.train()                          # modo entrenamiento
    pred_train = modelo(X_train_n)          # predicción
    loss_train = criterio(pred_train, y_train)  # calcular error
    
    # ── 2. BACKWARD: ajustar pesos ──
    optimizador.zero_grad()   # limpiar gradientes anteriores
    loss_train.backward()     # calcular gradientes (derivadas)
    optimizador.step()        # actualizar pesos
    
    # ── 3. VALIDACIÓN (sin gradientes) ──
    modelo.eval()
    with torch.no_grad():
        pred_val = modelo(X_val_n)
        loss_val = criterio(pred_val, y_val)
    
    # Guardar historial
    historial["train_loss"].append(loss_train.item())
    historial["val_loss"].append(loss_val.item())
    
    # Imprimir cada 20 épocas
    if epoca % 20 == 0 or epoca == 1:
        status = "✅" if loss_val.item() < 1.0 else "⏳"
        print(f"{epoca:<8} {loss_train.item():<14.4f} {loss_val.item():<14.4f} {status}")

print(f"\n🏁 Entrenamiento completado.")
print(f"   Loss final train: {historial['train_loss'][-1]:.4f}")
print(f"   Loss final val:   {historial['val_loss'][-1]:.4f}")

---
## 8. Curvas de pérdida y diagnóstico

In [ ]:
# Graficar curvas de pérdida

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epocas = list(range(1, n_epocas + 1))

# Plot completo
ax1.plot(epocas, historial["train_loss"], color="#3498db", linewidth=2, label="Train")
ax1.plot(epocas, historial["val_loss"],   color="#e74c3c", linewidth=2, label="Validation")
ax1.set_xlabel("Época", fontsize=12)
ax1.set_ylabel("Loss (MSE)", fontsize=12)
ax1.set_title("Curvas de pérdida (completas)", fontsize=12, fontweight="bold")
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.2)

# Plot últimas 100 épocas (zoom)
corte = max(0, n_epocas - 100)
ax2.plot(epocas[corte:], historial["train_loss"][corte:], color="#3498db", linewidth=2, label="Train")
ax2.plot(epocas[corte:], historial["val_loss"][corte:],   color="#e74c3c", linewidth=2, label="Validation")
ax2.set_xlabel("Época", fontsize=12)
ax2.set_ylabel("Loss (MSE)", fontsize=12)
ax2.set_title("Últimas 100 épocas (zoom)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

# Diagnóstico automático
train_final = historial["train_loss"][-1]
val_final   = historial["val_loss"][-1]
ratio = val_final / train_final if train_final > 0 else float('inf')

print("📊 Diagnóstico automático:")
if train_final > 2.0 and val_final > 2.0:
    print("  ⚠️ UNDERFITTING: ambas loss altas. Probá más neuronas o más épocas.")
elif ratio > 2.0:
    print("  ⚠️ OVERFITTING: val_loss >> train_loss. Probá menos neuronas o regularización.")
else:
    print("  ✅ BUEN BALANCE: train y val loss son cercanas y bajas.")
print(f"  Ratio val/train: {ratio:.2f} (ideal < 2.0)")

In [ ]:
# Evaluación final en test

modelo.eval()
with torch.no_grad():
    pred_test = modelo(X_test_n)

# Calcular métricas (reutilizando lo de Clase 6)
y_real_list = y_test.squeeze().tolist()
y_pred_list = pred_test.squeeze().tolist()

mae_val = sum(abs(r - p) for r, p in zip(y_real_list, y_pred_list)) / len(y_real_list)
mse_val = sum((r - p) ** 2 for r, p in zip(y_real_list, y_pred_list)) / len(y_real_list)
rmse_val = mse_val ** 0.5
media = sum(y_real_list) / len(y_real_list)
ss_res = sum((r - p) ** 2 for r, p in zip(y_real_list, y_pred_list))
ss_tot = sum((r - media) ** 2 for r in y_real_list)
r2_val = 1 - ss_res / ss_tot if ss_tot > 0 else 0

print("📊 Evaluación en TEST (datos nunca vistos):\n")
print(f"  MAE:  {mae_val:.4f}")
print(f"  RMSE: {rmse_val:.4f}")
print(f"  R²:   {r2_val:.4f}")

# Gráfico real vs predicho
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_real_list, y_pred_list, c="#3498db", s=60, edgecolors="#333", alpha=0.8)
lims = [min(y_real_list + y_pred_list) - 0.5, max(y_real_list + y_pred_list) + 0.5]
ax.plot(lims, lims, "--", color="red", linewidth=1.5, label="Predicción perfecta")
ax.set_xlabel("Nota real", fontsize=12)
ax.set_ylabel("Nota predicha", fontsize=12)
ax.set_title(f"Test set — MAE={mae_val:.3f}, R²={r2_val:.3f}", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

---
## 9. Experimentos con hiperparámetros

Los hiperparámetros son configuraciones que **vos elegís** (no se aprenden del entrenamiento).

| Hiperparámetro | Valores típicos | Efecto |
|---|---|---|
| Learning rate | 0.001, 0.01, 0.1 | Velocidad de aprendizaje |
| Épocas | 50, 100, 200, 500 | Cuánto entrena |
| Neuronas ocultas | 8, 16, 32, 64 | Capacidad del modelo |
| Capas ocultas | 1, 2, 3 | Profundidad del modelo |

In [ ]:
# Experimento: comparar diferentes learning rates

lrs = [0.1, 0.01, 0.001, 0.0001]
resultados_lr = {}

fig, axes = plt.subplots(1, len(lrs), figsize=(16, 4))

for ax, lr in zip(axes, lrs):
    torch.manual_seed(42)
    m = MLP(n_features=5, hidden1=16, hidden2=8)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.MSELoss()
    
    train_losses, val_losses = [], []
    
    for ep in range(200):
        m.train()
        pred = m(X_train_n)
        loss = crit(pred, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        m.eval()
        with torch.no_grad():
            vl = crit(m(X_val_n), y_val).item()
        
        train_losses.append(loss.item())
        val_losses.append(vl)
    
    ax.plot(train_losses, color="#3498db", linewidth=1.5, label="Train")
    ax.plot(val_losses,   color="#e74c3c", linewidth=1.5, label="Val")
    ax.set_title(f"LR = {lr}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Época")
    ax.set_ylim(0, max(5, max(val_losses[:50])))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    
    resultados_lr[lr] = val_losses[-1]

fig.suptitle("Efecto del Learning Rate en el entrenamiento",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\n📊 Val Loss final por Learning Rate:")
for lr, vl in sorted(resultados_lr.items(), key=lambda x: x[1]):
    mejor = " ← mejor" if vl == min(resultados_lr.values()) else ""
    print(f"  LR={lr:<8} Val Loss={vl:.4f}{mejor}")

In [ ]:
# Experimento: comparar diferentes tamaños de red

configs = [
    ("Pequeña (4,2)",   4,  2),
    ("Mediana (16,8)",  16, 8),
    ("Grande (64,32)",  64, 32),
    ("Enorme (128,64)", 128, 64),
]

fig, axes = plt.subplots(1, len(configs), figsize=(16, 4))

resultados_size = {}

for ax, (nombre, h1, h2) in zip(axes, configs):
    torch.manual_seed(42)
    m = MLP(n_features=5, hidden1=h1, hidden2=h2)
    n_p = sum(p.numel() for p in m.parameters())
    opt = torch.optim.Adam(m.parameters(), lr=0.01)
    crit = nn.MSELoss()
    
    train_losses, val_losses = [], []
    
    for ep in range(200):
        m.train()
        pred = m(X_train_n)
        loss = crit(pred, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        
        m.eval()
        with torch.no_grad():
            vl = crit(m(X_val_n), y_val).item()
        
        train_losses.append(loss.item())
        val_losses.append(vl)
    
    ax.plot(train_losses, color="#3498db", linewidth=1.5, label="Train")
    ax.plot(val_losses,   color="#e74c3c", linewidth=1.5, label="Val")
    ax.set_title(f"{nombre}\n({n_p} params)", fontsize=10, fontweight="bold")
    ax.set_xlabel("Época")
    ax.set_ylim(0, 5)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)
    
    resultados_size[nombre] = {"val_loss": val_losses[-1], "params": n_p}

fig.suptitle("Efecto del tamaño de la red en el entrenamiento",
             fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

print("\n📊 Resultados por tamaño de red:")
print(f"{'Config':<20} {'Params':<10} {'Val Loss':<12}")
print("─" * 42)
for nombre, r in resultados_size.items():
    print(f"{nombre:<20} {r['params']:<10} {r['val_loss']:<12.4f}")

print("\n💡 Más grande ≠ mejor. Un modelo más grande puede overfittear con pocos datos.")

---
## 10. Ejercicio evaluable

### Consigna

Modificá y entrenás tu propia variante del MLP.

### Criterio de aprobación

- ✅ Definiste un MLP con una arquitectura diferente a la de la clase
- ✅ Entrenaste al menos 100 épocas con curvas de train/val
- ✅ Calculaste MAE y R² en test
- ✅ Comparaste tu resultado con el modelo base de la clase

In [ ]:
# ══════════════════════════════════════════════════════════════
#  EJERCICIO EVALUABLE — Tu propia red neuronal
# ══════════════════════════════════════════════════════════════

# TODO: Definí tu propia arquitectura MLP
# Podés cambiar:
#   - Cantidad de neuronas en las capas ocultas
#   - Agregar una tercera capa oculta
#   - Cambiar ReLU por Tanh
#   - Cambiar el learning rate
#   - Cambiar la cantidad de épocas

class MiMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.red = nn.Sequential(
            # TODO: definí tus capas
            nn.Linear(5, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    
    def forward(self, x):
        return self.red(x)

# TODO: configurar entrenamiento
torch.manual_seed(42)
mi_modelo = MiMLP()
mi_lr     = 0.01        # TODO: elegí tu learning rate
mi_epocas = 200         # TODO: elegí cuántas épocas

mi_opt  = torch.optim.Adam(mi_modelo.parameters(), lr=mi_lr)
mi_crit = nn.MSELoss()

n_p = sum(p.numel() for p in mi_modelo.parameters())
print(f"Mi MLP: {n_p} parámetros")
print(mi_modelo)

In [ ]:
# TODO: Loop de entrenamiento
# Copiá la estructura del loop de la sección 7 y adaptalo a mi_modelo

mi_train_loss = []
mi_val_loss = []

for epoca in range(1, mi_epocas + 1):
    # TODO: forward + backward + step
    pass

print(f"Entrenamiento completado: {mi_epocas} épocas.")

In [ ]:
# TODO: Graficar curvas de pérdida de tu modelo
# Consejo: usá la misma estructura de gráfico de la sección 8

pass

In [ ]:
# TODO: Evaluar en test y comparar con el modelo base

mi_mae_test  = 0  # TODO: calculá MAE en test
mi_r2_test   = 0  # TODO: calculá R² en test

print("\n📊 Comparación:")
print(f"{'Modelo':<20} {'MAE test':<12} {'R² test'}")
print("─" * 40)
print(f"{'Base (16, 8)':<20} {mae_val:<12.4f} {r2_val:.4f}")
print(f"{'MI modelo':<20} {mi_mae_test:<12.4f} {mi_r2_test:.4f}")

if mi_r2_test > r2_val:
    print("\n🏆 ¡Tu modelo supera al base!")
else:
    print("\n📌 El modelo base es mejor. ¿Qué podrías cambiar?")

print("\n🎉 Guardá este notebook y envialo como entrega de la Clase 7.")

---
## Resumen de la clase

| Concepto | Lo que aprendiste |
|---|---|
| Neurona artificial | Suma ponderada + activación |
| Funciones de activación | ReLU (ocultas), Sigmoid (salida binaria), Tanh (alternativa) |
| MLP | Perceptrón multicapa: capas apiladas de neuronas |
| Loss | MSE para regresión, CrossEntropy para clasificación |
| Backpropagation | Forward → Loss → Backward → Update (repite) |
| Learning rate | Controla la velocidad del aprendizaje |
| Diagnóstico | Curvas train vs val para detectar over/underfitting |

### Próxima clase

En la **Clase 8** vamos a construir un **mini LLM desde cero** que genera texto carácter por carácter usando todo lo que aprendiste.

> 📌 **Recordá:** el 90% de entrenar redes neuronales es experimentar con hiperparámetros y leer las curvas de pérdida.